# Smart Wave Allocation for Pharmaceutical Distribution
## Deep Reinforcement Learning Prototype (PPO)

This notebook implements a simplified dynamic reinforcement learning model for smart wave allocation in pharmaceutical warehousing, following the KGDRL research paradigm **without** the knowledge guidance module.

**Problem**: Given a stream of incoming orders, dynamically group them into "waves" (batches) for coordinated picking, balancing:
- Picking path efficiency
- Temperature compliance (ambient/cool/cold/frozen)
- Delivery deadline satisfaction
- Wave capacity constraints

**Approach**: Markov Decision Process (MDP) solved via Proximal Policy Optimization (PPO)

In [ ]:
# =============================================================================
# 0. IMPORTS
# =============================================================================
import numpy as np
import random
import math
import collections
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

# PyTorch for DRL
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Libraries loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Data Generation: Empirical Distributions

Based on the casebook enterprise data:
- **Order size**: 3-5 items (discrete distribution)
- **Temperature**: Ambient 55%, Cool 25%, Cold 15%, Frozen 5%
- **Deadlines**: Standard 8-24h, Urgent 2-6h (20% urgent)
- **Arrivals**: Poisson process with configurable rate (default 3/min for prototype)
- **Warehouse**: 8 zones in a 2×4 grid

In [ ]:
# =============================================================================
# 1. DATA GENERATION - EMPIRICAL DISTRIBUTIONS
# =============================================================================

@dataclass
class SKU:
    """Individual stock keeping unit"""
    sku_id: int
    zone: int
    temp_category: int  # 0=ambient, 1=cool, 2=cold, 3=frozen

@dataclass
class Order:
    """Customer order"""
    order_id: int
    arrival_time: float
    skus: List[SKU]
    deadline: float  # hours from arrival
    volume: float
    is_urgent: bool

    @property
    def n_items(self):
        return len(self.skus)

    @property
    def dominant_temp(self):
        """Most frequent temperature category in order"""
        temps = [sku.temp_category for sku in self.skus]
        return max(set(temps), key=temps.count)

    @property
    def required_zones(self):
        return list(set(sku.zone for sku in self.skus))

    @property
    def zone_centroid(self):
        """Centroid of required zones"""
        zones = self.required_zones
        coords = [(z % 4, z // 4) for z in zones]
        return (np.mean([c[0] for c in coords]), np.mean([c[1] for c in coords]))


class DataGenerator:
    """
    Generates synthetic order data based on empirical distributions
    from the pharmaceutical distribution casebook.
    """

    def __init__(self, seed: int = 42, base_rate: float = 3.0):
        """
        Args:
            seed: random seed
            base_rate: average orders per minute (default 3.0 for prototype)
        """
        self.rng = np.random.RandomState(seed)
        random.seed(seed)

        self.n_zones = 8
        self.zone_coords = {z: (z % 4, z // 4) for z in range(self.n_zones)}

        # Temperature distribution
        self.temp_probs = [0.55, 0.25, 0.15, 0.05]
        self.temp_names = ['ambient', 'cool', 'cold', 'frozen']

        # Order size distribution
        self.order_sizes = [3, 4, 5]
        self.order_size_probs = [0.40, 0.35, 0.25]

        # Arrival process
        self.base_rate = base_rate
        self.peak_prob = 0.05
        self.peak_multiplier = 2.8

        # Deadline distribution
        self.urgent_prob = 0.20

    def _is_peak_period(self, t: float) -> bool:
        """Bimodal peak: 9AM=60min, 2PM=300min"""
        peak1 = abs(t - 60) < 30
        peak2 = abs(t - 300) < 30
        return peak1 or peak2

    def generate_orders(self, horizon_hours: float = 4.0) -> List[Order]:
        """Generate orders over a time horizon"""
        horizon_min = horizon_hours * 60
        orders = []
        order_id = 0
        t = 0.0

        while t < horizon_min:
            rate = self.base_rate
            if self._is_peak_period(t):
                rate *= 1.5
            if self.rng.rand() < self.peak_prob:
                rate *= self.peak_multiplier

            dt = self.rng.exponential(1.0 / rate)
            t += dt
            if t >= horizon_min:
                break

            order = self._generate_single_order(order_id, t)
            orders.append(order)
            order_id += 1

        return orders

    def _generate_single_order(self, order_id: int, arrival_time: float) -> Order:
        n_items = self.rng.choice(self.order_sizes, p=self.order_size_probs)

        skus = []
        for i in range(n_items):
            zone = self.rng.randint(0, self.n_zones)
            temp = self.rng.choice(4, p=self.temp_probs)
            skus.append(SKU(sku_id=order_id*100+i, zone=zone, temp_category=temp))

        is_urgent = self.rng.rand() < self.urgent_prob
        if is_urgent:
            deadline = self.rng.uniform(2, 6)
        else:
            deadline = self.rng.uniform(8, 24)

        volume = n_items * self.rng.uniform(5, 20)

        return Order(
            order_id=order_id,
            arrival_time=arrival_time,
            skus=skus,
            deadline=deadline,
            volume=volume,
            is_urgent=is_urgent
        )

    def zone_distance(self, z1: int, z2: int) -> float:
        """Manhattan distance between zones"""
        c1 = self.zone_coords[z1]
        c2 = self.zone_coords[z2]
        return abs(c1[0] - c2[0]) + abs(c1[1] - c2[1])


# Test data generation
gen = DataGenerator(seed=42, base_rate=3.0)
orders = gen.generate_orders(horizon_hours=2.0)
print(f"Generated {len(orders)} orders")
print(f"Avg items/order: {np.mean([o.n_items for o in orders]):.2f}")
print(f"Urgent orders: {sum(1 for o in orders if o.is_urgent)} ({sum(1 for o in orders if o.is_urgent)/len(orders)*100:.1f}%)")
print(f"Avg deadline: {np.mean([o.deadline for o in orders]):.1f}h")

# Show sample order
sample = orders[0]
print(f"\nSample Order #{sample.order_id}:")
print(f"  Arrival: {sample.arrival_time:.1f}min, Deadline: {sample.deadline:.1f}h, Urgent: {sample.is_urgent}")
print(f"  Items: {sample.n_items}, Zones: {sample.required_zones}, Temp: {gen.temp_names[sample.dominant_temp]}")

## 2. Environment: PharmaWaveEnv

The environment implements the MDP formulation:
- **State**: Current wave status + candidate orders + temporal stats
- **Action**: Select an order to add to the active wave, or close the wave
- **Reward**: Composite of picking efficiency, temperature compliance, deadline satisfaction, and setup cost

In [ ]:
# =============================================================================
# 2. ENVIRONMENT DEFINITION
# =============================================================================

class PharmaWaveEnv:
    """
    Smart Wave Allocation Environment
    State: Current wave status + order pool + temporal info
    Action: Select order to add to wave, or close wave
    Reward: Picking efficiency + compliance + timeliness
    """

    def __init__(self,
                 orders: List[Order],
                 max_wave_orders: int = 20,
                 max_wave_volume: float = 300,
                 picking_speed: float = 60.0,
                 setup_time: float = 10.0,
                 time_step: float = 1.0,
                 alpha_eff: float = 1.0,
                 alpha_temp: float = 100.0,
                 alpha_deadline: float = 50.0,
                 alpha_setup: float = 5.0,
                 k_candidates: int = 10):

        self.all_orders = orders
        self.max_wave_orders = max_wave_orders
        self.max_wave_volume = max_wave_volume
        self.picking_speed = picking_speed
        self.setup_time = setup_time
        self.time_step = time_step

        # Reward coefficients
        self.alpha_eff = alpha_eff
        self.alpha_temp = alpha_temp
        self.alpha_deadline = alpha_deadline
        self.alpha_setup = alpha_setup

        self.k_candidates = k_candidates
        self.n_zones = 8
        self.n_temps = 4

        # Internal state
        self.current_time = 0.0
        self.order_pool: List[Order] = []
        self.pending_orders: List[Order] = []
        self.waves: List[Dict] = []

        self.active_wave_orders: List[Order] = []
        self.active_wave_volume = 0.0
        self.active_wave_start_time = 0.0
        self.active_wave_zones: set = set()
        self.active_wave_temps: set = set()

        self.total_picking_distance = 0.0
        self.total_setup_time = 0.0
        self.deadline_misses = 0
        self.temp_violations = 0

        self.done = False
        self.step_count = 0

    def reset(self):
        """Reset environment to initial state"""
        self.current_time = 0.0
        self.order_pool = []
        self.pending_orders = sorted(self.all_orders, key=lambda o: o.arrival_time)
        self.waves = []

        self.active_wave_orders = []
        self.active_wave_volume = 0.0
        self.active_wave_start_time = 0.0
        self.active_wave_zones = set()
        self.active_wave_temps = set()

        self.total_picking_distance = 0.0
        self.total_setup_time = 0.0
        self.deadline_misses = 0
        self.temp_violations = 0

        self.done = False
        self.step_count = 0

        self._process_arrivals()
        return self.get_state()

    def _process_arrivals(self):
        """Move arrived orders from pending to pool"""
        newly_arrived = [o for o in self.pending_orders if o.arrival_time <= self.current_time]
        self.pending_orders = [o for o in self.pending_orders if o.arrival_time > self.current_time]
        self.order_pool.extend(newly_arrived)

    def _estimate_picking_distance(self, orders: List[Order]) -> float:
        """Estimate TSP tour length for picking orders (simplified nearest-neighbor)"""
        if not orders:
            return 0.0

        zones = list(set(z for o in orders for z in o.required_zones))
        if not zones:
            return 0.0

        entry = (0, 0)
        coords = [self._get_zone_coord(z) for z in zones]

        unvisited = set(range(len(coords)))
        current = entry
        total_dist = 0.0

        while unvisited:
            nearest = min(unvisited, key=lambda i: self._euclid(current, coords[i]))
            total_dist += self._euclid(current, coords[nearest])
            current = coords[nearest]
            unvisited.remove(nearest)

        total_dist += self._euclid(current, entry)
        return total_dist

    def _get_zone_coord(self, z: int) -> Tuple[float, float]:
        return (z % 4, z // 4)

    def _euclid(self, a, b):
        return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

    def _close_wave(self) -> Tuple[float, Dict]:
        """Close active wave, compute rewards/penalties"""
        if not self.active_wave_orders:
            return 0.0, {}

        distance = self._estimate_picking_distance(self.active_wave_orders)
        picking_time = distance / self.picking_speed
        self.total_setup_time += self.setup_time

        finish_time = self.current_time + self.setup_time + picking_time

        # Deadline penalties
        deadline_penalty = 0.0
        for o in self.active_wave_orders:
            allowed_finish = o.arrival_time + o.deadline * 60
            if finish_time > allowed_finish:
                hours_late = (finish_time - allowed_finish) / 60
                deadline_penalty += hours_late * self.alpha_deadline
                self.deadline_misses += 1

        # Temperature penalty
        temp_penalty = 0.0
        if len(self.active_wave_temps) > 1:
            temps = sorted(self.active_wave_temps)
            incompatible = False
            if (0 in temps or 1 in temps) and (2 in temps or 3 in temps):
                incompatible = True
            if 2 in temps and 3 in temps:
                incompatible = True
            if incompatible:
                temp_penalty = self.alpha_temp
                self.temp_violations += 1

        efficiency_reward = -self.alpha_eff * distance
        setup_cost = -self.alpha_setup

        total_reward = efficiency_reward + temp_penalty + deadline_penalty + setup_cost

        wave_info = {
            'orders': len(self.active_wave_orders),
            'volume': self.active_wave_volume,
            'distance': distance,
            'finish_time': finish_time,
            'temps': list(self.active_wave_temps),
            'zones': list(self.active_wave_zones)
        }

        self.total_picking_distance += distance
        self.waves.append(wave_info)

        self.active_wave_orders = []
        self.active_wave_volume = 0.0
        self.active_wave_zones = set()
        self.active_wave_temps = set()
        self.active_wave_start_time = self.current_time

        return total_reward, wave_info

    def _get_candidates(self) -> List[Order]:
        """Get top-K candidate orders from pool (by urgency)"""
        if not self.order_pool:
            return []

        def urgency(o: Order):
            remaining = (o.arrival_time + o.deadline * 60) - self.current_time
            return remaining

        sorted_pool = sorted(self.order_pool, key=urgency)
        return sorted_pool[:self.k_candidates]

    def get_state(self) -> Dict:
        """Return state representation as dictionary"""
        candidates = self._get_candidates()

        wave_features = np.array([
            len(self.active_wave_orders) / self.max_wave_orders,
            self.active_wave_volume / self.max_wave_volume,
            (self.current_time - self.active_wave_start_time) / 120 if self.active_wave_orders else 0,
            len(self.active_wave_zones) / self.n_zones
        ], dtype=np.float32)

        zone_mask = np.zeros(self.n_zones, dtype=np.float32)
        for z in self.active_wave_zones:
            zone_mask[z] = 1.0

        temp_mask = np.zeros(self.n_temps, dtype=np.float32)
        for t in self.active_wave_temps:
            temp_mask[t] = 1.0

        candidate_features = []
        for i in range(self.k_candidates):
            if i < len(candidates):
                o = candidates[i]
                remaining_time = ((o.arrival_time + o.deadline * 60) - self.current_time) / 60
                cx, cy = o.zone_centroid
                feat = [
                    o.n_items / 5.0,
                    remaining_time / 24.0,
                    o.dominant_temp / 3.0,
                    cx / 3.0,
                    cy / 1.0,
                    1.0 if o.is_urgent else 0.0
                ]
            else:
                feat = [0.0] * 6
            candidate_features.extend(feat)
        candidate_features = np.array(candidate_features, dtype=np.float32)

        urgent_count = sum(1 for o in self.order_pool
                          if (o.arrival_time + o.deadline * 60) - self.current_time < 120)
        global_stats = np.array([
            urgent_count / max(len(self.order_pool), 1),
            len(self.pending_orders) / max(len(self.all_orders), 1),
            len(self.order_pool) / 50.0
        ], dtype=np.float32)

        state_dict = {
            'wave': wave_features,
            'zone_mask': zone_mask,
            'temp_mask': temp_mask,
            'candidates': candidate_features,
            'global': global_stats,
            'n_candidates': len(candidates),
            'can_close': len(self.active_wave_orders) >= 1
        }

        return state_dict

    def get_state_vector(self, state_dict: Dict) -> np.ndarray:
        """Flatten state dict to vector"""
        parts = [
            state_dict['wave'],
            state_dict['zone_mask'],
            state_dict['temp_mask'],
            state_dict['candidates'],
            state_dict['global']
        ]
        return np.concatenate(parts)

    @property
    def state_dim(self) -> int:
        return 4 + self.n_zones + self.n_temps + 6 * self.k_candidates + 3

    @property
    def action_dim(self) -> int:
        return self.k_candidates + 1

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """
        Action: 0..k_candidates-1 = add candidate order to wave
                k_candidates = close wave
        """
        self.step_count += 1
        candidates = self._get_candidates()
        reward = 0.0
        info = {'action_type': 'none'}

        if action < len(candidates) and action >= 0:
            order = candidates[action]

            if (len(self.active_wave_orders) < self.max_wave_orders and
               self.active_wave_volume + order.volume <= self.max_wave_volume):

                self.active_wave_orders.append(order)
                self.active_wave_volume += order.volume
                self.active_wave_zones.update(order.required_zones)
                self.active_wave_temps.add(order.dominant_temp)
                self.order_pool.remove(order)

                reward = -0.1
                info['action_type'] = 'add'
            else:
                reward = -10.0
                info['action_type'] = 'invalid_capacity'

        elif action == len(candidates) or action == self.k_candidates:
            wave_reward, wave_info = self._close_wave()
            reward = wave_reward
            info = wave_info
            info['action_type'] = 'close'

        else:
            reward = -5.0
            info['action_type'] = 'invalid'

        self.current_time += self.time_step
        self._process_arrivals()

        if len(self.order_pool) == 0 and len(self.pending_orders) == 0:
            if self.active_wave_orders:
                wave_reward, wave_info = self._close_wave()
                reward += wave_reward
            self.done = True

        if self.step_count > 2000:
            if self.active_wave_orders:
                wave_reward, wave_info = self._close_wave()
                reward += wave_reward
            self.done = True

        state_dict = self.get_state()
        state_vec = self.get_state_vector(state_dict)

        return state_vec, reward, self.done, info

    def get_valid_actions(self, state_dict: Dict) -> List[int]:
        """Return list of valid action indices"""
        valid = []
        candidates = self._get_candidates()
        n_candidates = len(candidates)
        wave_order_full = len(self.active_wave_orders) >= self.max_wave_orders

        # Check each candidate individually for volume capacity
        if not wave_order_full:
            for i in range(min(n_candidates, self.k_candidates)):
                if self.active_wave_volume + candidates[i].volume <= self.max_wave_volume:
                    valid.append(i)

        # Closing wave (valid if wave has orders, pool empty, or capacity reached)
        can_close = len(self.active_wave_orders) >= 1
        if can_close or n_candidates == 0 or wave_order_full or len(valid) == 0:
            valid.append(self.k_candidates)

        return valid if valid else [self.k_candidates]


# Test environment
env = PharmaWaveEnv(orders, max_wave_orders=20)
state_dict = env.reset()
print(f"State dimension: {env.state_dim}")
print(f"Action dimension: {env.action_dim}")
print(f"Initial candidates: {state_dict['n_candidates']}")

# Run random policy
total_reward = 0
while not env.done:
    valid = env.get_valid_actions(env.get_state())
    action = random.choice(valid)
    state_vec, reward, done, info = env.step(action)
    total_reward += reward

print(f"\nRandom policy result:")
print(f"  Steps: {env.step_count}, Total reward: {total_reward:.1f}")
print(f"  Waves: {len(env.waves)}, Distance: {env.total_picking_distance:.1f}")
print(f"  Deadline misses: {env.deadline_misses}, Temp violations: {env.temp_violations}")

## 3. Rule-Based Heuristic Baselines

Before training DRL, we establish rule-based baselines:
- **FCFS**: First-come-first-serve
- **TEMP_FIRST**: Temperature-matching priority
- **ZONE_NN**: Zone nearest-neighbor
- **EDD**: Earliest due date
- **TZU**: Temperature-Zone-Urgency composite (our domain heuristic)

In [ ]:
# =============================================================================
# 3. RULE-BASED HEURISTICS
# =============================================================================

def run_heuristic(env: PharmaWaveEnv, heuristic_name: str, generator: DataGenerator) -> Dict:
    """Run a rule-based heuristic on the environment"""
    state_dict = env.reset()
    total_reward = 0.0

    while not env.done:
        candidates = env._get_candidates()
        valid_actions = env.get_valid_actions(state_dict)

        if not candidates or len(valid_actions) == 0:
            action = env.k_candidates
        else:
            if heuristic_name == 'FCFS':
                action = 0 if 0 in valid_actions else env.k_candidates

            elif heuristic_name == 'TEMP_FIRST':
                best_action = env.k_candidates
                if env.active_wave_temps:
                    for i, o in enumerate(candidates):
                        if i in valid_actions and o.dominant_temp in env.active_wave_temps:
                            best_action = i
                            break
                else:
                    if 0 in valid_actions:
                        best_action = 0
                action = best_action

            elif heuristic_name == 'ZONE_NN':
                if env.active_wave_zones:
                    wcx = np.mean([generator.zone_coords[z][0] for z in env.active_wave_zones])
                    wcy = np.mean([generator.zone_coords[z][1] for z in env.active_wave_zones])

                    best_dist = float('inf')
                    best_action = env.k_candidates
                    for i, o in enumerate(candidates):
                        if i in valid_actions:
                            ox, oy = o.zone_centroid
                            dist = abs(ox - wcx) + abs(oy - wcy)
                            if dist < best_dist:
                                best_dist = dist
                                best_action = i
                    action = best_action
                else:
                    action = 0 if 0 in valid_actions else env.k_candidates

            elif heuristic_name == 'EDD':
                best_action = env.k_candidates
                min_remaining = float('inf')
                for i, o in enumerate(candidates):
                    if i in valid_actions:
                        remaining = (o.arrival_time + o.deadline * 60) - env.current_time
                        if remaining < min_remaining:
                            min_remaining = remaining
                            best_action = i
                action = best_action

            elif heuristic_name == 'TZU':
                best_score = -float('inf')
                best_action = env.k_candidates

                for i, o in enumerate(candidates):
                    if i not in valid_actions:
                        continue

                    temp_match = 1.0 if o.dominant_temp in env.active_wave_temps or not env.active_wave_temps else 0.0

                    if env.active_wave_zones:
                        wcx = np.mean([generator.zone_coords[z][0] for z in env.active_wave_zones])
                        wcy = np.mean([generator.zone_coords[z][1] for z in env.active_wave_zones])
                        ox, oy = o.zone_centroid
                        zone_prox = 1.0 / (1.0 + abs(ox - wcx) + abs(oy - wcy))
                    else:
                        zone_prox = 1.0

                    remaining = (o.arrival_time + o.deadline * 60) - env.current_time
                    urgency = 1.0 / (1.0 + max(0, remaining / 60))

                    score = 0.4 * temp_match + 0.4 * zone_prox + 0.2 * urgency
                    if score > best_score:
                        best_score = score
                        best_action = i

                action = best_action
            else:
                action = random.choice(valid_actions)

        state_vec, reward, done, info = env.step(action)
        state_dict = env.get_state()
        total_reward += reward

    return {
        'total_reward': total_reward,
        'n_waves': len(env.waves),
        'total_distance': env.total_picking_distance,
        'deadline_misses': env.deadline_misses,
        'temp_violations': env.temp_violations,
        'waves': env.waves
    }


def compare_heuristics(generator: DataGenerator, n_instances=10):
    """Compare all heuristics on same instances"""
    heuristics = ['FCFS', 'TEMP_FIRST', 'ZONE_NN', 'EDD', 'TZU']
    all_results = {h: [] for h in heuristics}

    for i in tqdm(range(n_instances), desc="Evaluating heuristics"):
        orders = generator.generate_orders(horizon_hours=4.0)

        for h in heuristics:
            env = PharmaWaveEnv(orders, max_wave_orders=20)
            result = run_heuristic(env, h, generator)
            all_results[h].append(result)

    summary = {}
    for h in heuristics:
        rewards = [r['total_reward'] for r in all_results[h]]
        distances = [r['total_distance'] for r in all_results[h]]
        waves = [r['n_waves'] for r in all_results[h]]
        misses = [r['deadline_misses'] for r in all_results[h]]
        violations = [r['temp_violations'] for r in all_results[h]]

        summary[h] = {
            'avg_reward': np.mean(rewards),
            'std_reward': np.std(rewards),
            'avg_distance': np.mean(distances),
            'avg_waves': np.mean(waves),
            'avg_misses': np.mean(misses),
            'avg_violations': np.mean(violations)
        }

    return summary, all_results


# Run heuristic comparison
gen = DataGenerator(seed=42, base_rate=3.0)
heuristic_summary, heuristic_results = compare_heuristics(gen, n_instances=10)

print("\n" + "=" * 70)
print("HEURISTIC COMPARISON RESULTS (10 instances)")
print("=" * 70)
print(f"{'Heuristic':<12} {'Avg Reward':<12} {'Avg Dist':<12} {'Waves':<8} {'Misses':<8} {'Viol.':<8}")
print("-" * 70)
for h, stats in heuristic_summary.items():
    print(f"{h:<12} {stats['avg_reward']:<12.1f} {stats['avg_distance']:<12.1f} "
          f"{stats['avg_waves']:<8.1f} {stats['avg_misses']:<8.1f} {stats['avg_violations']:<8.1f}")
print("=" * 70)

## 4. PPO Deep Reinforcement Learning Agent

We implement PPO following the KGDRL architecture pattern (actor-critic with action masking) but **without** the knowledge guidance module.

**Network Architecture**:
- Actor: MLP(state_dim → hidden → hidden×2 → hidden → action_dim) → ReLU → Masked Softmax
- Critic: MLP(state_dim → hidden → hidden×2 → hidden → 1)

**Key Features**:
- Action masking for invalid actions
- Generalized Advantage Estimation (GAE)
- Clipped surrogate objective

In [ ]:
# =============================================================================
# 4. PPO DEEP REINFORCEMENT LEARNING
# =============================================================================

class PolicyNet(nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):
        super(PolicyNet, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim * 2)
        self.fc3 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return F.relu(self.fc4(x))


class ValueNet(nn.Module):
    def __init__(self, state_dim, hidden_dim):
        super(ValueNet, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim * 2)
        self.fc3 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)


class PPOAgent:
    def __init__(self, state_dim, action_dim, hidden_dim=None,
                 actor_lr=5e-4, critic_lr=1e-5, gamma=0.96, lmbda=0.95,
                 epochs=10, eps=0.2, device='cpu'):
        self.device = torch.device(device)
        self.state_dim = state_dim
        self.action_dim = action_dim
        hidden_dim = state_dim * 3 if hidden_dim is None else hidden_dim

        self.actor = PolicyNet(state_dim, hidden_dim, action_dim).to(self.device)
        self.critic = ValueNet(state_dim, hidden_dim).to(self.device)

        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

        self.gamma = gamma
        self.lmbda = lmbda
        self.epochs = epochs
        self.eps = eps

        self.actor_losses = []
        self.critic_losses = []

    def select_action(self, state, valid_actions, deterministic=False):
        state_tensor = torch.tensor(state, dtype=torch.float).unsqueeze(0).to(self.device)

        with torch.no_grad():
            probs_raw = self.actor(state_tensor).squeeze(0)

        mask = torch.zeros(self.action_dim, device=self.device)
        mask[valid_actions] = 1.0

        probs = probs_raw * mask
        if probs.sum() < 1e-8:
            probs = mask / mask.sum()
        else:
            probs = probs / probs.sum()

        if deterministic:
            action = torch.argmax(probs).item()
        else:
            dist = torch.distributions.Categorical(probs)
            action = dist.sample().item()

        log_prob = torch.log(probs[action] + 1e-8)
        return action, log_prob.item(), probs.cpu().numpy()

    def compute_advantages(self, rewards, values, next_values, dones):
        advantages = []
        gae = 0.0

        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * next_values[t] * (1 - dones[t]) - values[t]
            gae = delta + self.gamma * self.lmbda * gae * (1 - dones[t])
            advantages.insert(0, gae)

        return np.array(advantages)

    def update(self, trajectory):
        states = np.array([t['state'] for t in trajectory])
        actions = np.array([t['action'] for t in trajectory])
        old_log_probs = np.array([t['log_prob'] for t in trajectory])
        rewards = np.array([t['reward'] for t in trajectory])
        next_states = np.array([t['next_state'] for t in trajectory])
        dones = np.array([t['done'] for t in trajectory])
        valid_masks = [t['valid_actions'] for t in trajectory]

        with torch.no_grad():
            states_t = torch.tensor(states, dtype=torch.float).to(self.device)
            next_states_t = torch.tensor(next_states, dtype=torch.float).to(self.device)
            values = self.critic(states_t).squeeze().cpu().numpy()
            next_values = self.critic(next_states_t).squeeze().cpu().numpy()

        advantages = self.compute_advantages(rewards, values, next_values, dones)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        td_target = advantages + values

        states_t = torch.tensor(states, dtype=torch.float).to(self.device)
        actions_t = torch.tensor(actions, dtype=torch.long).view(-1, 1).to(self.device)
        old_log_probs_t = torch.tensor(old_log_probs, dtype=torch.float).view(-1, 1).to(self.device)
        advantages_t = torch.tensor(advantages, dtype=torch.float).view(-1, 1).to(self.device)
        td_target_t = torch.tensor(td_target, dtype=torch.float).view(-1, 1).to(self.device)

        for _ in range(self.epochs):
            probs_raw = self.actor(states_t)

            batch_size = probs_raw.shape[0]
            probs = torch.zeros_like(probs_raw)
            for i in range(batch_size):
                mask = torch.zeros(self.action_dim, device=self.device)
                mask[valid_masks[i]] = 1.0
                masked = probs_raw[i] * mask
                if masked.sum() < 1e-8:
                    probs[i] = mask / mask.sum()
                else:
                    probs[i] = masked / masked.sum()

            log_probs = torch.log(probs.gather(1, actions_t) + 1e-8)
            ratio = torch.exp(log_probs - old_log_probs_t)

            surr1 = ratio * advantages_t
            surr2 = torch.clamp(ratio, 1 - self.eps, 1 + self.eps) * advantages_t
            actor_loss = -torch.min(surr1, surr2).mean()

            critic_loss = F.mse_loss(self.critic(states_t), td_target_t)

            self.actor_optimizer.zero_grad()
            self.critic_optimizer.zero_grad()
            actor_loss.backward()
            critic_loss.backward()

            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)

            self.actor_optimizer.step()
            self.critic_optimizer.step()

            self.actor_losses.append(actor_loss.item())
            self.critic_losses.append(critic_loss.item())

print("PPO Agent class defined.")

## 5. Training Loop

The training loop follows the standard RL episodic pattern:
1. Generate a new problem instance (orders for a shift)
2. Run the agent's policy to collect a trajectory
3. Periodically update the policy using collected experience
4. Track metrics across episodes

In [ ]:
# =============================================================================
# 5. TRAINING LOOP
# =============================================================================

def train_ppo(generator: DataGenerator, n_episodes=100, update_interval=10,
              hidden_dim=128, actor_lr=5e-4, critic_lr=1e-5):
    """Train PPO agent"""

    # Infer dimensions
    orders = generator.generate_orders(horizon_hours=4.0)
    env = PharmaWaveEnv(orders)
    state_dim = env.state_dim
    action_dim = env.action_dim

    agent = PPOAgent(state_dim, action_dim, hidden_dim=hidden_dim,
                     actor_lr=actor_lr, critic_lr=critic_lr)

    episode_rewards = []
    episode_metrics = []

    for episode in tqdm(range(n_episodes), desc="Training PPO"):
        orders = generator.generate_orders(horizon_hours=4.0)
        env = PharmaWaveEnv(orders, max_wave_orders=20)
        state_dict = env.reset()
        state = env.get_state_vector(state_dict)

        trajectory = []
        episode_reward = 0.0

        while not env.done:
            valid_actions = env.get_valid_actions(state_dict)
            action, log_prob, probs = agent.select_action(state, valid_actions)

            next_state_vec, reward, done, info = env.step(action)
            next_state_dict = env.get_state()

            trajectory.append({
                'state': state,
                'action': action,
                'log_prob': log_prob,
                'reward': reward,
                'next_state': next_state_vec,
                'done': float(done),
                'valid_actions': valid_actions
            })

            episode_reward += reward
            state = next_state_vec
            state_dict = next_state_dict

        episode_rewards.append(episode_reward)
        episode_metrics.append({
            'reward': episode_reward,
            'n_waves': len(env.waves),
            'distance': env.total_picking_distance,
            'misses': env.deadline_misses,
            'violations': env.temp_violations
        })

        # Update after collecting enough experience
        if len(trajectory) > 0 and (episode + 1) % update_interval == 0:
            agent.update(trajectory)

    return agent, episode_metrics


# Train the agent
gen_train = DataGenerator(seed=42, base_rate=3.0)
agent, metrics = train_ppo(gen_train, n_episodes=100, update_interval=10,
                           hidden_dim=128, actor_lr=5e-4, critic_lr=1e-5)

print(f"\nTraining complete! Final avg reward (last 10): {np.mean([m['reward'] for m in metrics[-10:]]):.1f}")

## 6. Evaluation and Visualization

We evaluate the trained PPO agent against the heuristic baselines on hold-out instances and visualize training progress.

In [ ]:
# =============================================================================
# 6. EVALUATION AND VISUALIZATION
# =============================================================================

def evaluate_agent(agent, generator: DataGenerator, n_eval=20, deterministic=True):
    """Evaluate agent on hold-out instances"""
    results = []

    for i in range(n_eval):
        orders = generator.generate_orders(horizon_hours=4.0)
        env = PharmaWaveEnv(orders, max_wave_orders=20)
        state_dict = env.reset()

        total_reward = 0.0

        while not env.done:
            valid_actions = env.get_valid_actions(state_dict)
            state = env.get_state_vector(state_dict)
            action, _, _ = agent.select_action(state, valid_actions, deterministic=deterministic)

            state_vec, reward, done, info = env.step(action)
            state_dict = env.get_state()
            total_reward += reward

        results.append({
            'total_reward': total_reward,
            'n_waves': len(env.waves),
            'total_distance': env.total_picking_distance,
            'deadline_misses': env.deadline_misses,
            'temp_violations': env.temp_violations,
            'avg_wave_size': np.mean([w['orders'] for w in env.waves]) if env.waves else 0
        })

    return results


# Evaluate trained PPO agent
gen_eval = DataGenerator(seed=123, base_rate=3.0)
ppo_results = evaluate_agent(agent, gen_eval, n_eval=20, deterministic=True)

ppo_reward = np.mean([r['total_reward'] for r in ppo_results])
ppo_dist = np.mean([r['total_distance'] for r in ppo_results])
ppo_waves = np.mean([r['n_waves'] for r in ppo_results])
ppo_misses = np.mean([r['deadline_misses'] for r in ppo_results])
ppo_viol = np.mean([r['temp_violations'] for r in ppo_results])

print("\n" + "=" * 70)
print("FINAL EVALUATION (20 hold-out instances)")
print("=" * 70)
print(f"{'Method':<12} {'Avg Reward':<12} {'Avg Dist':<12} {'Waves':<8} {'Misses':<8} {'Viol.':<8}")
print("-" * 70)
for h, stats in heuristic_summary.items():
    print(f"{h:<12} {stats['avg_reward']:<12.1f} {stats['avg_distance']:<12.1f} "
          f"{stats['avg_waves']:<8.1f} {stats['avg_misses']:<8.1f} {stats['avg_violations']:<8.1f}")
print(f"{'PPO (ours)':<12} {ppo_reward:<12.1f} {ppo_dist:<12.1f} "
      f"{ppo_waves:<8.1f} {ppo_misses:<8.1f} {ppo_viol:<8.1f}")
print("=" * 70)

In [ ]:
# =============================================================================
# 7. PLOTTING
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

rewards = [m['reward'] for m in metrics]
waves = [m['n_waves'] for m in metrics]
distances = [m['distance'] for m in metrics]
misses = [m['misses'] for m in metrics]

# Moving average
window = max(1, len(rewards) // 20)
if len(rewards) >= window:
    ma_rewards = np.convolve(rewards, np.ones(window)/window, mode='valid')
else:
    ma_rewards = rewards

axes[0, 0].plot(rewards, alpha=0.3, color='blue', label='Raw')
if len(ma_rewards) < len(rewards):
    axes[0, 0].plot(range(window-1, len(rewards)), ma_rewards, color='red', linewidth=2, label=f'MA({window})')
else:
    axes[0, 0].plot(ma_rewards, color='red', linewidth=2)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].set_title('Training Reward (PPO)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(waves, color='green')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Number of Waves')
axes[0, 1].set_title('Waves per Episode')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(distances, color='orange')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Total Picking Distance')
axes[1, 0].set_title('Picking Distance per Episode')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(misses, color='red')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Deadline Misses')
axes[1, 1].set_title('Deadline Misses per Episode')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 8. COMPARATIVE BAR CHART
# =============================================================================

methods = list(heuristic_summary.keys()) + ['PPO']
avg_rewards = [heuristic_summary[m]['avg_reward'] for m in methods[:-1]] + [ppo_reward]
avg_distances = [heuristic_summary[m]['avg_distance'] for m in methods[:-1]] + [ppo_dist]
avg_misses = [heuristic_summary[m]['avg_misses'] for m in methods[:-1]] + [ppo_misses]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#e377c2']

axes[0].bar(methods, avg_rewards, color=colors)
axes[0].set_ylabel('Average Total Reward')
axes[0].set_title('Total Reward Comparison')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(methods, avg_distances, color=colors)
axes[1].set_ylabel('Average Picking Distance')
axes[1].set_title('Picking Distance Comparison')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(methods, avg_misses, color=colors)
axes[2].set_ylabel('Average Deadline Misses')
axes[2].set_title('Deadline Misses Comparison')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insight: PPO learns to balance wave size (distance efficiency) with deadline/temperature constraints.")

## 9. Analysis and Interpretation

### What the DRL agent learns

1. **Wave sizing**: The agent learns optimal wave sizes — not too small (high setup cost) and not too large (inefficient picking paths)

2. **Temperature awareness**: Through the penalty signal, the agent implicitly learns to segregate temperature-incompatible orders

3. **Deadline prioritization**: The reward structure encourages urgent orders to be assigned to earlier waves

4. **Zone clustering**: The agent learns to group orders requiring nearby zones

### Why KGDRL (with knowledge guidance) could improve further

- **TZU heuristic injection**: Initialize the policy with Temperature-Zone-Urgency priors to accelerate learning
- **Graph structure**: Encode warehouse topology as a graph for GNN-based state representation
- **Hierarchical actions**: First select temperature category, then zone, then specific order

### Alternative: Dynamic Inventory Management

If the online stochastic nature of order arrivals makes wave allocation too difficult for DRL convergence, the same PPO framework can be redirected to **dynamic inventory replenishment** with:
- State = inventory levels + pipeline + demand forecast
- Action = replenishment quantities
- Reward = -(holding + stockout + ordering costs)


In [ ]:
# =============================================================================
# 10. SENSITIVITY ANALYSIS (Optional)
# =============================================================================

def run_sensitivity(param_name, param_values, n_eval=10):
    """Run sensitivity analysis on a parameter"""
    results = {}

    for val in param_values:
        gen = DataGenerator(seed=42, base_rate=3.0)
        orders = gen.generate_orders(horizon_hours=4.0)

        if param_name == 'alpha_temp':
            env = PharmaWaveEnv(orders, alpha_temp=val)
        elif param_name == 'alpha_deadline':
            env = PharmaWaveEnv(orders, alpha_deadline=val)
        elif param_name == 'max_wave_orders':
            env = PharmaWaveEnv(orders, max_wave_orders=val)
        else:
            env = PharmaWaveEnv(orders)

        # Run TZU heuristic
        result = run_heuristic(env, 'TZU', gen)
        results[val] = result

    return results


# Example: sensitivity to temperature penalty
temp_penalties = [10, 50, 100, 200, 500]
sens_results = run_sensitivity('alpha_temp', temp_penalties, n_eval=5)

print("\nSensitivity to Temperature Penalty (TZU heuristic):")
print(f"{'Penalty':<10} {'Reward':<12} {'Dist':<12} {'Misses':<8} {'Viol.':<8}")
print("-" * 50)
for val, res in sens_results.items():
    print(f"{val:<10} {res['total_reward']:<12.1f} {res['total_distance']:<12.1f} "
          f"{res['deadline_misses']:<8} {res['temp_violations']:<8}")

print("\nHigher temperature penalties reduce violations but may increase distance/misses.")